In [5]:
import geopandas as gpd
import zipfile
import tempfile
import os
import pandas as pd
from shapely.geometry import GeometryCollection, Point, MultiPoint

In [3]:
kmz_dir = "../../../data/shapes/"
output_path = f"{kmz_dir}/training_samples_2010.shp"

In [6]:
def normalize_geometries(gdf):
    rows = []

    for _, row in gdf.iterrows():
        geom = row.geometry

        if isinstance(geom, Point):
            rows.append(row)

        elif isinstance(geom, MultiPoint):
            for pt in geom.geoms:
                new_row = row.copy()
                new_row.geometry = pt
                rows.append(new_row)

        elif isinstance(geom, GeometryCollection):
            for g in geom.geoms:
                if isinstance(g, Point):
                    new_row = row.copy()
                    new_row.geometry = g
                    rows.append(new_row)

        else:
            # silently skip unsupported geometries
            pass

    return gpd.GeoDataFrame(rows, crs=gdf.crs)


In [7]:
gdfs = []

# ==========================
# LOOP THROUGH KMZ FILES
# ==========================
for file in os.listdir(kmz_dir):
    if file.lower().endswith(".kmz"):

        class_id = os.path.splitext(file)[0]  # e.g. "1", "2", "3"
        kmz_path = os.path.join(kmz_dir, file)

        with tempfile.TemporaryDirectory() as tmpdir:
            with zipfile.ZipFile(kmz_path, 'r') as kmz:
                kmz.extractall(tmpdir)

            # Find KML inside KMZ
            kml_files = [f for f in os.listdir(tmpdir) if f.endswith(".kml")]
            if not kml_files:
                continue

            kml_path = os.path.join(tmpdir, kml_files[0])

            # Read KML
            gdf = gpd.read_file(kml_path, driver="KML")
            gdf = normalize_geometries(gdf)
            # Add class column
            gdf["class_id"] = int(class_id)

            gdfs.append(gdf)

# ==========================
# MERGE ALL
# ==========================
merged_gdf = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True),
    crs=gdfs[0].crs
)

# ==========================
# OPTIONAL: REPROJECT
# ==========================
# merged_gdf = merged_gdf.to_crs(epsg=32736)  # example UTM

# ==========================
# SAVE SHAPEFILE
# ==========================
merged_gdf.to_file(output_path, driver="ESRI Shapefile")

print("Training shapefile created successfully.")

Training shapefile created successfully.


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15696\573990846.py:47: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  merged_gdf.to_file(output_path, driver="ESRI Shapefile")
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'description' to 'descriptio'
  ogr_write(
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field timestamp created as String field, though DateTime requested.
  ogr_write(
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field begin created as String field, though DateTime requested.
  ogr_write(
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field end created as String field, though DateTime requested.
  ogr_write(
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: Runtim